# STEP 01. 개발환경 확인
## 작업 계획
- Notebook이 .venv의 Python으로 실행되는지 확인한다.
- 필요한 패키지가 설치되어 있는지 확인한다.
## 이번에 하지 않는 것
- 크롤링 코드 작성

In [5]:
import sys
import platform

print("Python:", sys.version)
print("실행 위치:", sys.executable)   # .venv가 들어있어야 정상
print("Platform:", platform.platform())

Python: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
실행 위치: c:\dev\claude-code-agent-course\chapter11\ax-job-agent\.venv\Scripts\python.exe
Platform: Windows-11-10.0.26200-SP0


In [6]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

print("pandas:", pd.__version__)
print("requests:", requests.__version__)
print("BeautifulSoup import: OK")

pandas: 3.0.6
requests: 2.34.2
BeautifulSoup import: OK


# STEP 02. 수집 데이터 명세
## 작업 계획
- 수집할 채용공고 데이터의 컬럼(열)을 미리 정한다. (기준: `docs/SPEC.md` 6장)
- 컬럼 이름을 `COLUMNS` 리스트로 만들고, 9개가 맞는지 출력해서 확인한다.
## 이번에 하지 않는 것
- 크롤링, 인터넷 요청, 패키지 설치

**DataFrame의 한 행 = 채용공고 한 건**

| 컬럼 | 의미 | 예시 |
|---|---|---|
| `company_name` | 회사명 | ㈜에이아이랩 |
| `job_title` | 공고 제목 | 생성형 AI 엔지니어 채용 |
| `career` | 경력 조건 | 신입·경력 3년↑ |
| `location` | 근무 지역 | 서울 강남구 |
| `posted_date` | 등록일 | 2026-09-20 |
| `closing_date` | 마감일 | 2026-10-10 / 상시채용 |
| `job_url` | 공고 URL (**고유 키**) | https://www.jobkorea.co.kr/... |
| `search_keyword` | 어떤 검색어로 찾았는지 | LLM |
| `collected_at` | 수집 시각 | 2026-09-23 10:00 |

In [7]:
COLUMNS = [
    "company_name",    # 회사명
    "job_title",       # 공고 제목
    "career",          # 경력 조건
    "location",        # 근무 지역
    "posted_date",     # 등록일
    "closing_date",    # 마감일
    "job_url",         # 공고 URL (고유 키)
    "search_keyword",  # 어떤 검색어로 찾았는지
    "collected_at",    # 수집 시각
]

In [8]:
print("컬럼 개수:", len(COLUMNS))   # 9가 나와야 정상

for i, col in enumerate(COLUMNS, start=1):
    print(f"{i}. {col}")

컬럼 개수: 9
1. company_name
2. job_title
3. career
4. location
5. posted_date
6. closing_date
7. job_url
8. search_keyword
9. collected_at


## 실행 결과 해석
- 성공 여부: 성공
- 확인한 내용: 컬럼 9개가 SPEC.md 6장과 같은 순서로 출력됨 (company_name ~ collected_at)
- 예상과 다른 부분: 없음
- 다음 단계 진행 가능 여부: 가능

# STEP 03. 채용공고 페이지 접근 테스트
## 작업 계획
- 검색어 1개(`LLM`)로 잡코리아 검색 결과 페이지에 **딱 1번** 요청을 보낸다.
- 상태 코드, 응답 형식, 응답 길이를 보고 페이지에 접근할 수 있는지 확인한다.
- HTML 안에 검색어가 들어 있는지 확인한다. (데이터 추출은 다음 STEP에서 한다.)
## robots.txt 확인 결과
- `User-agent: *` 규칙에서 `/Search/` 경로는 막혀 있지 않다.
- AI 학습용 크롤러는 사이트 전체가 차단되어 있다. (이 프로젝트는 학습용 수집이 아니다.)
- 요청은 1번만 보낸다. (반복 요청이나 여러 페이지 요청은 하지 않는다.)
- 접근이 막히면 **우회하지 않고** 샘플 데이터로 전환한다.
## 이번에 하지 않는 것
- 반복 요청, 여러 페이지 요청, 상세 페이지 요청
- HTML에서 데이터 추출

In [9]:
KEYWORD = "LLM"   # 검색어 (처음에는 1개만 사용)

SEARCH_URL = "https://www.jobkorea.co.kr/Search/"   # 잡코리아 검색 페이지
params = {"stext": KEYWORD}   # 주소 뒤에 ?stext=LLM 으로 붙는 값

headers = {
    # 일반 브라우저(Chrome)처럼 보이는 User-Agent
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/128.0.0.0 Safari/537.36"
    ),
}

In [10]:
# 요청은 딱 1번만 보낸다 (이 셀을 여러 번 실행하지 않기)
response = requests.get(SEARCH_URL, params=params, headers=headers, timeout=10)

print("요청한 주소:", response.url)

요청한 주소: https://www.jobkorea.co.kr/Search/?stext=LLM


In [11]:
html = response.text

print("상태 코드:", response.status_code)   # 200이면 정상
print("Content-Type:", response.headers.get("Content-Type"))   # text/html 이면 정상
print("응답 길이(글자 수):", len(html))
print(f"HTML 안에 '{KEYWORD}' 등장 횟수:", html.count(KEYWORD))   # 0이면 검색 결과가 없거나 막힌 것일 수 있음
print("-" * 50)
print(html[:500])   # HTML 앞부분 500자

상태 코드: 200
Content-Type: text/html; charset=utf-8
응답 길이(글자 수): 340027
HTML 안에 'LLM' 등장 횟수: 191
--------------------------------------------------
<!DOCTYPE html><html lang="ko" data-sentry-component="RootLayout" data-sentry-source-file="layout.tsx"><head translate="no"><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1, maximum-scale=1, user-scalable=no"/><link rel="preload" as="image" href="https://file2.jobkorea.co.kr/Net/Mng/Image/LogoImage?FN=2026/08/박정민캠페인최종본.png"/><link rel="stylesheet" href="https://frontend-app-cdn.jobkorea.co.kr/jobs/_next/static/css/49ef98337cc62b42.css" data-precedence="ne


## 실행 결과 해석
- 요청 성공 여부: 성공 (상태 코드 200, text/html)
- 확인한 데이터: 응답 길이 340,027자, HTML 안에 'LLM'이 191번 등장 → 검색 결과가 HTML에 들어 있음
- 예상과 다른 부분: 없음 (차단되지 않음)
- 다음 단계 진행 가능 여부: 가능 → STEP 04에서 공고 데이터 추출